In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

In [2]:
# coarse labels（粗颗粒度）
coarse_labels = [
    {
        "label_name": "制动系统-抖动",
        "label_text": "制动系统相关抖动，通常表现为踩刹车时方向盘或车身振动，高速制动时更明显。"
    },
    {
        "label_name": "轮胎/底盘-振动",
        "label_text": "轮胎或底盘相关振动，通常表现为高速行驶时方向盘抖动、车身发抖，可能与动平衡或轮胎磨损有关。"
    },
    {
        "label_name": "转向系统-异动",
        "label_text": "转向系统问题，表现为方向盘抖动、跑偏、转向间隙异常或异响。"
    },
    {
        "label_name": "发动机-异响",
        "label_text": "发动机相关异响，可能出现在冷启动、怠速或加速时。"
    },
    {
        "label_name": "车载显示系统-异常",
        "label_text": "中控屏或仪表显示异常，例如黑屏、卡死、显示模糊、白屏或系统无响应（display malfunction）。"
    },
    {
        "label_name": "车窗系统-故障",
        "label_text": "车窗升降或控制异常，例如车窗无法升降、卡住、自动升降失灵或异响。"
    },
        {
        "label_name": "车身结构-A柱问题",
        "label_text": "车辆A柱相关问题，例如异响、漏水、风噪或结构松动，通常在行驶或雨天时出现。"
    },
    {
        "label_name": "空调系统-异常",
        "label_text": "车内空调系统问题，例如不制冷、不制热、风量异常、有异味或噪音（AC malfunction）。"
    },
    {
        "label_name": "导航系统-故障",
        "label_text": "车载导航系统异常，例如定位不准、路线错误、卡顿或无法使用（navigation failure）。"
    },
        {
        "label_name": "氛围灯系统-异常",
        "label_text": "车内氛围灯异常，例如不亮、颜色不对、闪烁或局部失效（ambient light malfunction）。"
    },
    {
        "label_name": "蓝牙连接-问题",
        "label_text": "车载蓝牙连接异常，例如无法连接、频繁断连、通话或音乐异常（bluetooth issue）。"
    },
    {
        "label_name": "中控系统-功能异常",
        "label_text": "中控系统功能异常，例如按钮失灵、操作卡顿、控制逻辑错误或系统无响应。"
    },
    {
        "label_name": "车载充电系统-故障",
        "label_text": "车内充电相关问题，例如USB接口无电、充电慢或无线充电失效（charging failure）。"
    },
    {
        "label_name": "车身外观-保险杠问题",
        "label_text": "保险杠相关问题，例如松动、变形、异响或安装不良，可能影响外观或行驶安全。"
    },
]



# fine labels（细颗粒度）
fine_labels = [
    {
        "label_id": "540SV01",
        "label_name": "前制动盘变形-制动抖动",
        "label_text": "前制动盘变形，常见现象是高速或踩刹车时方向盘明显抖动，制动时更严重。"
    },
    {
        "label_id": "540SV02",
        "label_name": "前轮动平衡异常-高速抖动",
        "label_text": "前轮动平衡异常，常见现象是高速行驶时方向盘抖动，但踩刹车未必明显加重。"
    },
    {
        "label_id": "540SV03",
        "label_name": "刹车片磨损不均-制动振动",
        "label_text": "刹车片磨损不均匀，常导致制动时振动或异响。"
    },
    {
        "label_id": "540SV04",
        "label_name": "转向拉杆间隙过大-方向盘振动",
        "label_text": "转向拉杆间隙过大，可能导致方向盘振动、转向松旷或异响。"
    },
    {
        "label_id": "540SV05",
        "label_name": "正时链条异响-冷启动敲击声",
        "label_text": "冷启动时发动机舱出现敲击异响，热车后减弱，可能与正时链条有关。"
    },
    # ==================== 中控显示 ============================
    {
        "label_id": "540DS01",
        "label_name": "中控屏死机/卡顿-display halted",
        "label_text": "中控显示屏卡住、无响应或完全死机（halted），触控无反应，需要重启系统。"
    },
    {
        "label_id": "540DS02",
        "label_name": "显示屏黑屏/无法点亮-display malfunction",
        "label_text": "中控或仪表屏无法点亮、黑屏或间歇性失效，属于显示系统故障（display malfunction）。"
    },
    {
        "label_id": "540DS03",
        "label_name": "显示屏发白/发雾-milky display",
        "label_text": "显示屏画面发白、模糊、有雾状或泛白现象（milky display），影响正常查看。"
    },
    {
        "label_id": "540DS04",
        "label_name": "屏幕闪烁/花屏-display glitch",
        "label_text": "显示屏闪烁、花屏、画面异常跳动或显示错乱，可能是系统或硬件问题。"
    },
       # ================ 车窗 =======================
    {
        "label_id": "540WS01",
        "label_name": "车窗无法升降-window stuck",
        "label_text": "车窗无法升起或降下，按钮操作无效，可能卡住或电机故障。"
    },
    {
        "label_id": "540WS02",
        "label_name": "车窗升降缓慢/卡顿",
        "label_text": "车窗升降速度变慢或过程中卡顿，不顺畅，可能与轨道或电机有关。"
    },
    {
        "label_id": "540WS03",
        "label_name": "车窗自动升降失灵-auto window failure",
        "label_text": "车窗一键升降功能失效，需要持续按住按钮，自动控制异常。"
    },
    {
        "label_id": "540WS04",
        "label_name": "车窗异响-window noise",
        "label_text": "车窗升降过程中出现异响，如摩擦声、卡顿声或杂音。"
    },
        # ================= A柱 =================
    {
        "label_id": "540AP01",
        "label_name": "A柱异响-行驶中吱响",
        "label_text": "车辆行驶过程中A柱位置出现异响，如吱吱声或塑料摩擦声，颠簸路面更明显。"
    },
    {
        "label_id": "540AP02",
        "label_name": "A柱漏水-雨天渗水",
        "label_text": "雨天或洗车后A柱位置出现渗水或水迹，可能与密封不良有关。"
    },
    {
        "label_id": "540AP03",
        "label_name": "A柱风噪-高速呼啸声",
        "label_text": "高速行驶时A柱附近出现明显风噪或呼啸声，影响车内静音。"
    },

    # ================= 空调 =================
    {
        "label_id": "540AC01",
        "label_name": "空调不制冷-AC not cooling",
        "label_text": "开启空调后无冷风或制冷效果很差，可能与冷媒或压缩机有关（AC not cooling）。"
    },
    {
        "label_id": "540AC02",
        "label_name": "空调不制热-AC no heating",
        "label_text": "开启暖风后没有热风或温度不足，可能与加热系统或水温有关。"
    },
    {
        "label_id": "540AC03",
        "label_name": "空调异味-AC smell",
        "label_text": "空调出风口有异味，如霉味或刺鼻气味，通常与滤芯或蒸发箱有关。"
    },
    {
        "label_id": "540AC04",
        "label_name": "空调风量异常-weak airflow",
        "label_text": "空调风量很小或不稳定，即使调到最大档位也风力不足。"
    },
    {
        "label_id": "540AC05",
        "label_name": "空调噪音大-AC noise",
        "label_text": "空调运行时出现明显噪音，如嗡嗡声或异响，可能来自风机或压缩机。"
    },

    # ================= 导航 =================
    {
        "label_id": "540NV01",
        "label_name": "导航定位不准-GPS inaccurate",
        "label_text": "导航定位偏移或漂移，车辆位置与实际不一致（GPS inaccurate）。"
    },
    {
        "label_id": "540NV02",
        "label_name": "导航卡顿/无响应-navigation lag",
        "label_text": "导航界面卡顿、加载缓慢或无响应，影响使用体验。"
    },
    {
        "label_id": "540NV03",
        "label_name": "导航路线错误-wrong routing",
        "label_text": "导航推荐路线不合理或明显错误，例如绕路或无法规划正确路径。"
    },
    {
        "label_id": "540NV04",
        "label_name": "导航无法启动-navigation failure",
        "label_text": "导航系统无法打开或直接崩溃，属于系统故障（navigation failure）。"
    },
    # ================= 氛围灯 =================
    {
        "label_id": "540AL01",
        "label_name": "氛围灯不亮-ambient light off",
        "label_text": "车内氛围灯完全不亮或部分区域失效（ambient light off），夜间明显。"
    },
    {
        "label_id": "540AL02",
        "label_name": "氛围灯颜色异常-wrong color",
        "label_text": "氛围灯颜色显示异常，与设置不一致或出现错色（wrong color）。"
    },
    {
        "label_id": "540AL03",
        "label_name": "氛围灯闪烁-flickering light",
        "label_text": "氛围灯出现闪烁、不稳定亮度或忽明忽暗现象（flickering）。"
    },

    # ================= 蓝牙 =================
    {
        "label_id": "540BT01",
        "label_name": "蓝牙无法连接-bluetooth cannot connect",
        "label_text": "手机无法与车机蓝牙连接，搜索不到设备或连接失败。"
    },
    {
        "label_id": "540BT02",
        "label_name": "蓝牙频繁断连-bluetooth disconnect",
        "label_text": "蓝牙连接不稳定，频繁自动断开（disconnect）或需要反复连接。"
    },
    {
        "label_id": "540BT03",
        "label_name": "蓝牙通话异常-call issue",
        "label_text": "蓝牙通话有杂音、延迟或对方听不清，影响通话质量。"
    },
    {
        "label_id": "540BT04",
        "label_name": "蓝牙音乐卡顿-audio lag",
        "label_text": "蓝牙播放音乐时出现卡顿、延迟或断续（audio lag）。"
    },

    # ================= 中控 =================
    {
        "label_id": "540MC01",
        "label_name": "中控按键失灵-button failure",
        "label_text": "中控区域按键无反应或操作失效，无法正常控制功能。"
    },
    {
        "label_id": "540MC02",
        "label_name": "中控系统卡顿-system lag",
        "label_text": "中控系统操作卡顿、反应慢或切换界面延迟（system lag）。"
    },
    {
        "label_id": "540MC03",
        "label_name": "中控死机/重启-system crash",
        "label_text": "中控系统突然死机或自动重启，功能中断（system crash）。"
    },

    # ================= 充电 =================
    {
        "label_id": "540CH01",
        "label_name": "USB接口无电-no power",
        "label_text": "车内USB接口无法供电，插入设备无反应（no power）。"
    },
    {
        "label_id": "540CH02",
        "label_name": "充电速度慢-slow charging",
        "label_text": "设备充电速度明显偏慢，达不到正常充电水平。"
    },
    {
        "label_id": "540CH03",
        "label_name": "无线充电失效-wireless charging failure",
        "label_text": "无线充电功能无法使用，设备放置后无充电反应。"
    },

    # ================= 保险杠 =================
    {
        "label_id": "540BM01",
        "label_name": "保险杠松动-loose bumper",
        "label_text": "保险杠安装不牢固，有松动或晃动现象（loose bumper）。"
    },
    {
        "label_id": "540BM02",
        "label_name": "保险杠异响-bumper noise",
        "label_text": "行驶过程中保险杠区域出现异响，可能与固定件有关。"
    },
    {
        "label_id": "540BM03",
        "label_name": "保险杠变形-damaged bumper",
        "label_text": "保险杠出现变形、开裂或外观损伤（damaged bumper）。"
    },
]

In [3]:
# Loading Model
# top-k（embedding）
embedding_model_name = "BAAI/bge-small-zh-v1.5"
embedder = SentenceTransformer(embedding_model_name)

# rerank
reranker_model_name = "BAAI/bge-reranker-base"
reranker = CrossEncoder(reranker_model_name)


# FAISS Index
def build_faiss_index(label_items):
    texts = [x["label_text"] for x in label_items]
    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim) 
    index.add(embeddings)
    return index, embeddings

In [4]:
coarse_index, _ = build_faiss_index(coarse_labels)
fine_index, _ = build_faiss_index(fine_labels)


# 4. top-k definition
def retrieve_top_k(query_text, label_items, index, top_k=3):
    query_emb = embedder.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    scores, indices = index.search(query_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        item = dict(label_items[idx])  
        item["retrieval_score"] = float(score)
        results.append(item)
    return results


# 5. rerank 
def rerank_candidates(query_text, candidates, top_k=5):
    pairs = [[query_text, c["label_text"]] for c in candidates]
    rerank_scores = reranker.predict(pairs)

    results = []
    for c, s in zip(candidates, rerank_scores):
        item = dict(c)
        item["rerank_score"] = float(s)

        # To be modified
        item["final_score"] = 0.3 * item["retrieval_score"] + 0.7 * item["rerank_score"]
        results.append(item)

    results.sort(key=lambda x: x["final_score"], reverse=True)
    return results[:top_k]


# pipeline
def predict_tags(query_text, retrieve_k=5, return_k=3):
    # coarse top-k retrieval
    coarse_candidates = retrieve_top_k(
        query_text=query_text,
        label_items=coarse_labels,
        index=coarse_index,
        top_k=retrieve_k
    )

    # fine top-k retrieval
    fine_candidates = retrieve_top_k(
        query_text=query_text,
        label_items=fine_labels,
        index=fine_index,
        top_k=retrieve_k
    )

    # rerank
    coarse_ranked = rerank_candidates(query_text, coarse_candidates, top_k=return_k)
    fine_ranked = rerank_candidates(query_text, fine_candidates, top_k=return_k)

    return {
        "query": query_text,
        "coarse_topk": coarse_ranked,
        "fine_topk": fine_ranked
    }

In [19]:
# Testing
if __name__ == "__main__":
    query = 'RN_preapproval_82/82.60_氛围灯颜色无法调节_CCB，{"l11": "氛围灯颜色无法调节"， "l12": "客户2024年10月24日在店内加装原厂香氛系统后抱怨 ，调节氛围灯选择颜色时选择其他颜色会立刻跳回第一种颜色深海蓝，无法选择其他颜色。氛围灯颜色会自动多色彩转换。"， "l13": "无"， "l14": 此车辆在外加装了仪表台空调出风口氛围灯，在店内加装了原厂香氛。"， "l15": "故障一直存在影响车辆正常使用"， "l16": "无"， "l17": "主机故障"}'
    result = predict_tags(query_text=query, retrieve_k=4, return_k=3)

    print("输入文本：", result["query"])
    print("\n=== Most Revelant Coarse Labels ===")
    for i, item in enumerate(result["coarse_topk"], 1):
        print(
            f"{i}. {item['label_name']} | "
            f"retrieval={item['retrieval_score']:.4f}, "
            f"rerank={item['rerank_score']:.4f}, "
            f"final={item['final_score']:.4f}"
        )

    print("\n=== Most Revelant Fine Labels ===")
    for i, item in enumerate(result["fine_topk"], 1):
        print(
            f"{i}. {item['label_name']} | "
            f"retrieval={item['retrieval_score']:.4f}, "
            f"rerank={item['rerank_score']:.4f}, "
            f"final={item['final_score']:.4f}"
        )

输入文本： RN_preapproval_82/82.60_氛围灯颜色无法调节_CCB，{"l11": "氛围灯颜色无法调节"， "l12": "客户2024年10月24日在店内加装原厂香氛系统后抱怨 ，调节氛围灯选择颜色时选择其他颜色会立刻跳回第一种颜色深海蓝，无法选择其他颜色。氛围灯颜色会自动多色彩转换。"， "l13": "无"， "l14": 此车辆在外加装了仪表台空调出风口氛围灯，在店内加装了原厂香氛。"， "l15": "故障一直存在影响车辆正常使用"， "l16": "无"， "l17": "主机故障"}

=== Most Revelant Coarse Labels ===
1. 氛围灯系统-异常 | retrieval=0.7221, rerank=0.3792, final=0.4821
2. 空调系统-异常 | retrieval=0.5445, rerank=0.1952, final=0.3000
3. 车载显示系统-异常 | retrieval=0.5070, rerank=0.1186, final=0.2351

=== Most Revelant Fine Labels ===
1. 氛围灯颜色异常-wrong color | retrieval=0.7272, rerank=0.7743, final=0.7602
2. 氛围灯不亮-ambient light off | retrieval=0.6741, rerank=0.6060, final=0.6264
3. 氛围灯闪烁-flickering light | retrieval=0.5953, rerank=0.4190, final=0.4719


## Sample Cases

  E_E300L_VIN_Shanghai Star_电话微信无法接听，2024-7-18 姓氏，您好，针对您所反馈的车辆相关事宜，
  授权经销商已与您联系并诚邀您到店检测，届时授权经销商会结合您车辆检测结论提供对应建议方案或予以您详细解释说明，若后续您对此有其它疑虑，
  可直接联系授权经销商沟通，感谢您的理解与支持。 2024年7月17姓氏代公司首次投诉：客户于2023年1月1日在上海利星汽车维修有限公司购买E300L车辆，
  客户表示情况如下： 从提车开始就出现蓝牙连接后电话微信接听双方听不见声音的故障，通话时间显示为黄色，需要切换手机再切换回蓝牙连接才能恢复，
  频率非常高大概10次有3-4会出现，去店内检查3次了不是没有回复就是检查不出问题，头两天去店内检测给到回复是微信是软件兼容问题，
  车机接听没问题蓝牙有问题可能是手机有问题，但是换了一个其他手机连接还是会出现此故障，店内也录了视频，此故障已经严重影响行车安全，要求解决问题。 
  针对客户所述情况，坐席已安抚解释。请相关工作人员在2H内联系客户并按照规范要求将相应进展内容更新至系统中，谢谢。

RN_preapproval_82/82.60_氛围灯颜色无法调节_CCB，{"l11": "氛围灯颜色无法调节"， "l12": "客户2024年10月24日在店内加装原厂香氛系统后抱怨 ，
调节氛围灯选择颜色时选择其他颜色会立刻跳回第一种颜色深海蓝，无法选择其他颜色。氛围灯颜色会自动多色彩转换。"， "l13": "无"， "l14": 
"此车辆在外加装了仪表台空调出风口氛围灯，在店内加装了原厂香氛。"， "l15": "故障一直存在影响车辆正常使用"， "l16": "无"， "l17": "主机故障"}

"RS_TI_42/42.00_后制动片脱落，{l11: 车辆起步不了，加大油门咯砰一声，行驶过程中咯吱咯吱，节奏性的摩擦声 ，左后刹车盘上面有异物一块;， 
l12: # 客户反馈，启动车辆后，挂挡走，发现无法行驶，加大油门后，后部砰一声，可以正常行驶，行驶过程中，发现后部有节奏的摩擦声，不敢开，联系我店，
拖车进店。# 技师检查发现，两后制动片脱落， l13: 无相关维修历史和服务措施， l14: 无加装改装， l15: 故障一直存在，目前无法行驶， 
l16: 无相关SI LI TPT文件， l17: 1.制动片本身故障2.雨水导致制动盘、片粘连后导致损坏}"